In [1]:
import pandas as pd
import numpy as np

print("Loading feature engineered dataset...")

df = pd.read_csv(
    "feature_engineered_transactions.csv"
)

print(df.shape)

df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Sorted.")

Loading feature engineered dataset...
(2713386, 26)
Sorted.


In [2]:
df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Sorted by timestamp.")

Sorted by timestamp.


In [6]:
print("="*60)
print("BUILDING NODE MAPPING")
print("="*60)

all_wallets = pd.concat([
    df["from_address"],
    df["to_address"]
]).unique()

wallet_to_id = {
    wallet: idx
    for idx, wallet in enumerate(all_wallets)
}

df["src"] = df["from_address"].map(wallet_to_id)
df["dst"] = df["to_address"].map(wallet_to_id)

print(f"Unique wallets: {len(wallet_to_id):,}")

print(df[[
    "from_address",
    "to_address",
    "src",
    "dst"
]].head())

BUILDING NODE MAPPING
Unique wallets: 333,077
                                 from_address  \
0  0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5   
1  0x366E99753E62D9BeAb40537cC9B2beCDBD9105C0   
2  0xdA9890CbC6982950508Aa947C196f3A93636E47C   
3  0xE72EB31b59F85b19499A0F3b3260011894FA0d65   
4  0x62223651d6a33D58Be70Eb9876c3CaF7096169ef   

                                   to_address  src    dst  
0  0x8072566Fcdbe9E1827C57A8091085a3095bd4cF0    0  34592  
1  0x42F7c5275aC4372156027d939843C9C42523DF2E    1   2456  
2  0x42F7c5275aC4372156027d939843C9C42523DF2E    2   2456  
3  0xCa96163F13d48F73D994951ebF1134397914aDFe    3  58569  
4  0xDf6e87287c96498320EC42a02C57A59C78c5133F    4    485  


In [7]:
edge_features = [

    "log_transaction_value",

    "log_src_tx_count_past",
    "log_dst_tx_count_past",

    "log_src_to_dst_count_past",
    "log_dst_to_src_count_past",

    "log_time_since_src_last",
    "log_time_since_dst_last",

    "log_src_value_sum_past",
    "log_dst_value_sum_past"
]

print(edge_features)
print(f"Total Features: {len(edge_features)}")

['log_transaction_value', 'log_src_tx_count_past', 'log_dst_tx_count_past', 'log_src_to_dst_count_past', 'log_dst_to_src_count_past', 'log_time_since_src_last', 'log_time_since_dst_last', 'log_src_value_sum_past', 'log_dst_value_sum_past']
Total Features: 9


In [8]:
src = df["src"].astype(np.int64).values

dst = df["dst"].astype(np.int64).values

timestamps = (
    df["timestamp"]
    .astype(np.int64)
    .values
)

labels = (
    df["is_wash_trading"]
    .astype(np.int64)
    .values
)

edge_feat = (
    df[edge_features]
    .astype(np.float32)
    .values
)

In [9]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_mask = np.arange(n) < train_end

val_mask = (
    (np.arange(n) >= train_end)
    &
    (np.arange(n) < val_end)
)

test_mask = (
    np.arange(n) >= val_end
)

In [10]:
print("="*60)
print("GRAPH DATASET SUMMARY")
print("="*60)

print(f"Nodes          : {len(np.unique(np.concatenate([src,dst]))):,}")
print(f"Edges          : {len(src):,}")
print(f"Fraud Edges    : {labels.sum():,}")
print(f"Fraud Ratio    : {labels.mean():.4%}")

print()

print(f"Train          : {train_mask.sum():,}")
print(f"Validation     : {val_mask.sum():,}")
print(f"Test           : {test_mask.sum():,}")

print()

print(f"Train Fraud    : {labels[train_mask].sum():,}")
print(f"Val Fraud      : {labels[val_mask].sum():,}")
print(f"Test Fraud     : {labels[test_mask].sum():,}")

GRAPH DATASET SUMMARY
Nodes          : 333,077
Edges          : 2,713,386
Fraud Edges    : 18,953
Fraud Ratio    : 0.6985%

Train          : 1,899,370
Validation     : 407,008
Test           : 407,008

Train Fraud    : 15,818
Val Fraud      : 1,684
Test Fraud     : 1,451


In [13]:
np.savez_compressed(

    "nft_graph_dataset.npz",

    src=src,
    dst=dst,

    timestamps=timestamps,

    edge_feat=edge_feat,

    labels=labels,

    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

print("Graph dataset saved.")

Graph dataset saved.
